In [1]:
import sys
import os
import copy
project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

In [2]:
%load_ext autoreload
%autoreload 2
import os
import json
import pickle as pkl
from gbmhackathon.utils import global_wrapper

/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [3]:
path_config="../gbmhackathon/utils/config.json"
path_config="../gbmhackathon/utils/config_2_XG_boost.json"

with open(path_config, 'r') as f:
    config= json.load(f)

print(config.keys())

output_dir = "/home/sagemaker-user/results"
os.makedirs(output_dir, exist_ok=True)
config['global_settings']["output_dir"]="/home/sagemaker-user/results"
config['global_settings']["device"]='cuda'
config["MME_Model"]["training"]["epochs"]=3
config["MME_Model"]["architecture"]["MME"]["wes_cfg"]["net_config"]["layers"]= [1790, 550, 64]
del config["MME_Model"]["modalities_data"]["modalities"]["spatial"] ## Marche pas à cause de timothée
#del config["MME_Model"]["modalities_data"]["modalities"]["wes"] ## Marche pas à cause de timothée
config["gbm_head"]["head_cfg"]["net_config"]["layers"]=[6,1024,512,5]
print(config)
config["Experiments"]["Multiple_run_experiment"]["params"]["n_run"]=5

dict_keys(['global_settings', 'MME_Model', 'gbm_head', 'Global_Architecture', 'Experiments'])
{'global_settings': {'device': 'cuda', 'output_dir': '/home/sagemaker-user/results'}, 'MME_Model': {'modalities_data': {'modalities': {'hne': 'embeddings_HnE_OptimusH0.pkl', 'clinical': '2025-03-30_14-23_clinical_emb_V1.pkl', 'wes': '2025-04-05_13-40_wes_emb_V1.pkl'}, 'pkl_storage_folder': 'embedding_V1', 'missing_mods': 's3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/missing_mod_per_samples.pkl'}, 'training': {'batch_size': 16, 'epochs': 3, 'InfoNCE_Loss': {'temperature': 10, 'similarity': 'nt-xent', 'alpha': 0, 'bound': -50, 'beta': 0.2, 'nce_eps': 1e-08, 'reg_eps': 1e-08, 'slope': 0.1, 'rate': -5, 'use_all_positives': False}, 'Optimizer': {'lr': 0.001}}, 'architecture': {'MME': {'hne_cfg': {'net_type': 'mlp', 'net_config': {'layers': [1536, 512, 64], 'dropout': 0.3, 'act_fn': 'torch.nn.ReLU', 'norm_layer': 'torch.nn.LayerNorm'}}, 'spatial_cfg': {'net_type': 'atten

In [4]:


config_cp=copy.deepcopy(config)
config_cp["MME_Model"]["training"]["epochs"]=0
config_cp2=copy.deepcopy(config)
youhou=global_wrapper.ModularModel(config_cp) ### Pas bien de faire comme ça mais au moins y'a pas d'entrainement phase 1

['hne', 'clinical', 'wes']
hne_cfg
['hne', 'clinical', 'wes']
spatial_cfg
['hne', 'clinical', 'wes']
wes_cfg
['hne', 'clinical', 'wes']
clinical_cfg
['hne', 'clinical', 'wes']
bulk_cfg
['hne', 'clinical', 'wes']
sc_cfg


In [5]:
youhou.fit()

start phase 1 ...
Using device : cuda
device MME : cuda
hne torch.Size([16, 1536])
clinical torch.Size([16, 12])
wes torch.Size([16, 1790])
{'net_type': 'mlp', 'net_config': {'layers': [1536, 512, 64], 'dropout': 0.3, 'act_fn': <class 'torch.nn.modules.activation.ReLU'>, 'norm_layer': <class 'torch.nn.modules.normalization.LayerNorm'>}, 'device': 'cuda'}
Using device: cuda
No potential residual connections found
Using device: cuda
No potential residual connections found
Using device: cuda
No potential residual connections found
start phase 2 ...
cas 2 : device reconnu : cuda 
Using device: cuda


/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter beta not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate', 'smoothing_func'].
  warnings.warn(
/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter nce_eps not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate', 'smoothing_func'].
  warnings.warn(
/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter reg_eps not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate', 'smoothing_func'].
  warnings.warn(


predictive loader (X_dict) device : cuda:0
predictive loader (batch_targets) device : cuda:0
ok!!!
using : XGBOOST
cas 2 : device reconnu : cuda 
Using device: cuda
starting phase 2 for target :  0
starting phase 2 for target :  1
starting phase 2 for target :  2
starting phase 2 for target :  3
entrinement et cross validation...
Matrice de confusion :
 [[ 0 20]
 [ 0 94]]

Rapport de classification :
               precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        20
         1.0       0.82      1.00      0.90        94

    accuracy                           0.82       114
   macro avg       0.41      0.50      0.45       114
weighted avg       0.68      0.82      0.75       114

[1. 0. 1. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 0. 0. 0. 0.
 0. 1. 1. 0. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 0. 1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 

/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(avera

Matrice de confusion :
 [[94  0]
 [20  0]]

Rapport de classification :
               precision    recall  f1-score   support

         0.0       0.82      1.00      0.90        94
         1.0       0.00      0.00      0.00        20

    accuracy                           0.82       114
   macro avg       0.41      0.50      0.45       114
weighted avg       0.68      0.82      0.75       114

[0. 1. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 1. 1.
 1. 0. 0. 1. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(avera

In [6]:


config_cp2["MME_Model"]["training"]["epochs"]=100
print(config_cp2)

youhou2=global_wrapper.ModularModel(config_cp2)

{'global_settings': {'device': 'cuda', 'output_dir': '/home/sagemaker-user/results'}, 'MME_Model': {'modalities_data': {'modalities': {'hne': 'embeddings_HnE_OptimusH0.pkl', 'clinical': '2025-03-30_14-23_clinical_emb_V1.pkl', 'wes': '2025-04-05_13-40_wes_emb_V1.pkl'}, 'pkl_storage_folder': 'embedding_V1', 'missing_mods': 's3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/missing_mod_per_samples.pkl'}, 'training': {'batch_size': 16, 'epochs': 100, 'InfoNCE_Loss': {'temperature': 10, 'similarity': 'nt-xent', 'alpha': 0, 'bound': -50, 'beta': 0.2, 'nce_eps': 1e-08, 'reg_eps': 1e-08, 'slope': 0.1, 'rate': -5, 'use_all_positives': False}, 'Optimizer': {'lr': 0.001}}, 'architecture': {'MME': {'hne_cfg': {'net_type': 'mlp', 'net_config': {'layers': [1536, 512, 64], 'dropout': 0.3, 'act_fn': 'torch.nn.ReLU', 'norm_layer': 'torch.nn.LayerNorm'}}, 'spatial_cfg': {'net_type': 'attention', 'net_config': {'dim': 1790, 'depth': 4, 'num_heads': 8, 'mlp_ratio': 4.0, 'qkv_bias':

In [7]:
youhou2.fit()

start phase 1 ...
Using device : cuda
device MME : cuda
hne torch.Size([16, 1536])
clinical torch.Size([16, 12])
wes torch.Size([16, 1790])
{'net_type': 'mlp', 'net_config': {'layers': [1536, 512, 64], 'dropout': 0.3, 'act_fn': <class 'torch.nn.modules.activation.ReLU'>, 'norm_layer': <class 'torch.nn.modules.normalization.LayerNorm'>}, 'device': 'cuda'}
Using device: cuda
No potential residual connections found
Using device: cuda
No potential residual connections found
Using device: cuda
No potential residual connections found


/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter beta not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate', 'smoothing_func'].
  warnings.warn(
/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter nce_eps not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate', 'smoothing_func'].
  warnings.warn(
/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:25: ParameterNotFoundWarning: Parameter reg_eps not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate', 'smoothing_func'].
  warnings.warn(


EPOCH 0 LOSS: 8.5127
EPOCH 1 LOSS: 8.4931
EPOCH 2 LOSS: 8.5874
EPOCH 3 LOSS: 8.4995
EPOCH 4 LOSS: 8.4951
EPOCH 5 LOSS: 8.5041
EPOCH 6 LOSS: 8.5666
EPOCH 7 LOSS: 8.5001
EPOCH 8 LOSS: 8.5353
EPOCH 9 LOSS: 8.5462
EPOCH 10 LOSS: 8.3993
EPOCH 11 LOSS: 8.5290
EPOCH 12 LOSS: 8.5324
EPOCH 13 LOSS: 8.4645
EPOCH 14 LOSS: 8.5391
EPOCH 15 LOSS: 8.4707
EPOCH 16 LOSS: 8.4608
EPOCH 17 LOSS: 8.5264
EPOCH 18 LOSS: 8.3695
EPOCH 19 LOSS: 8.3639
EPOCH 20 LOSS: 8.5034
EPOCH 21 LOSS: 8.4378
EPOCH 22 LOSS: 8.5047
EPOCH 23 LOSS: 8.5103
EPOCH 24 LOSS: 8.4920
EPOCH 25 LOSS: 8.5146
EPOCH 26 LOSS: 8.5132
EPOCH 27 LOSS: 8.4970
EPOCH 28 LOSS: 8.4910
EPOCH 29 LOSS: 8.4940
EPOCH 30 LOSS: 8.5238
EPOCH 31 LOSS: 8.4932
EPOCH 32 LOSS: 8.4898
EPOCH 33 LOSS: 8.4736
EPOCH 34 LOSS: 8.5023
EPOCH 35 LOSS: 8.5024
EPOCH 36 LOSS: 8.4947
EPOCH 37 LOSS: 8.4195
EPOCH 38 LOSS: 8.5076
EPOCH 39 LOSS: 8.3371
EPOCH 40 LOSS: 8.4333
EPOCH 41 LOSS: 8.4916
EPOCH 42 LOSS: 8.4855
EPOCH 43 LOSS: 8.4985
EPOCH 44 LOSS: 8.4922
EPOCH 45 LOSS: 8.484